# Milestone 12 — unified advanced movement research (Google Colab GPU)

**NFL Big Data Bowl 2027 · Combine → NFL movement transfer.** One notebook replaces the eight Milestone 12 notebooks (12.0, 12A–12G). It orchestrates; the algorithms live in `nflbdb.research.advanced_movement.m12` (unchanged) and the infrastructure in `nflbowl.research` (stage runners, Drive artifacts, reports). Consolidation decisions: `docs/nflbowl/M12_MERGE_MATRIX.md`. How to run: `docs/nflbowl/M12_COLAB_EXECUTION_GUIDE.md`.

## Part 0 · Research overview

**Competition objective.** Turn player-tracking data into decision-relevant insight. Here: does what an athlete shows in the Combine drills (tracked at 10 Hz) carry information about how he moves in NFL games, beyond the official stopwatch tests (M1 = position, draft year, height, weight, official forty, 10-yd split)?

**Research questions (declared before execution, `m12.config.TRACKS`).**

| Q | Track | Question | Primary comparison |
|---|---|---|---|
| Q3 | 12C | Are NFL speed-conditioned acceleration quantiles more reliable than maxima / linear envelopes, and do Combine speed features predict them beyond official tests? | reliability C1–C4 vs C0a/C0b; M3 vs M1, M2 vs M1 (Holm) |
| Q2 | 12B | Could unreliable Combine measurement mask athlete-level relationships? | RC (B1) vs naive (B0) held out; corrected estimands |
| Q1 | 12A | Does the whole Combine a(v) function carry information that scalar bands lose? | A2 (FPCA) vs A0 (bands) on the NFL function |
| Q4 | 12D | Do several drills jointly reveal traits more reproducible than single drill metrics? | factor vs best single feature test–retest |
| Q5 | 12E | Do whole movement distributions recover cross-domain similarity that point features miss? | E5 conditional OT vs E0 retrieval AUC |
| Q6 | 12F | Does an encoder trained on Combine AND NFL sequences carry transferable physics? | inductive F4 vs F0 / random; M1+F4 vs M1 |
| Q7 | 12G | Which advanced method beats simple physical models on identical athletes? | common-cohort tournament vs M1 |

**Execution order = dependency order** (`fd.dependency_manifest()`): 12.0 → 12C → 12B → 12A → 12D → 12E → 12F → 12G. 12C comes first among the methods because it FREEZES the NFL target (`nfl.q90_shared`) that 12A, 12F and 12G use; so the method parts below are numbered in that order (Part 5 = 12C … Part 9 = 12E).

**Novelty of the framework.** (i) A target frozen by a declared reliability rule on NFL data only, before any Combine value is joined; (ii) functional, measurement-error, latent-factor, optimal-transport and self-supervised representations compared on ONE fold manifest, ONE grid and ONE ladder; (iii) every comparison paired on identical athletes with athlete-bootstrap intervals and declared negative controls; (iv) transductive diagnostics and inductive prediction kept apart.

**Decision relevance.** Scouts: comparables and reliable physical profiles. General managers: whether tracked Combine information adds anything beyond the stopwatch (the tournament). Coaches / player development: reproducible descriptions of game acceleration capacity. Part 12 maps each track's evidence to these uses — only where the data support it.

```mermaid
flowchart LR
  subgraph Inputs
    B[(Mode A: M12F bundle<br/>sequence store + folds)]
    W[(Mode B: research warehouse<br/>M11 foundation + NFL histograms)]
  end
  W --> F0[12.0 foundation<br/>grids, cohorts, folds]
  F0 --> C[12C distributional capacity<br/>frozen NFL target]
  F0 --> BM[12B measurement error]
  C --> A[12A functional transfer]
  F0 --> D[12D multi-view traits]
  F0 --> E[12E conditional OT]
  B --> F[12F shared encoder on GPU]
  C -. target .-> F
  A --> EV[Part 11 unified evaluation]
  BM --> EV
  C --> EV
  D --> EV
  E --> EV
  F --> EV
  EV --> G[12G synthesis + tournament]
  G --> R[Decisions + final report on Drive]
```

**Two input modes.** Mode A (portable M12F bundle) supports 12F and the bundle EDA only. Mode B (research warehouse on Drive) supports 12.0 and 12A–12E, 12G. Every section declares its inputs; a section whose inputs are missing is skipped with the reason, it never crashes the notebook.

**Run all** works top to bottom. Nothing expensive starts without its configuration flag (Part 2). Every figure, table, dataset, checkpoint and report is written to Google Drive under `RUN_DIR`.

## Part 1 · Google Colab configuration

1. *Runtime → Change runtime type → GPU* (T4 is enough for F4).
2. Run 1.1 once per runtime. If `nflbowl` had been imported before in this runtime, *Runtime → Restart session* after 1.1 and continue at 1.2.
3. Colab's CUDA build of PyTorch is never replaced: the package is installed with `--no-deps`, and its other dependencies are installed explicitly (without torch / numpy pins).

In [ ]:
# Part 1.1 — install nflbowl / nflbdb from GitHub (torch untouched)
INSTALL_POT = False      # optional: exact W2 for 12E (E2); without POT, E2 is reported not_available (never substituted)
INSTALL_PYMC = False     # optional: 12B Bayesian latent model (B4, minutes to hours)
%pip install -q 'polars>=1.10' 'pyarrow>=15' 'psutil>=5.9' 'pyyaml>=6.0' 'matplotlib>=3.8' 'duckdb>=1.1' markdown
%pip install -q --upgrade --force-reinstall --no-deps 'git+https://github.com/mybenkhadda/NFL_bowl.git@main'
if INSTALL_POT:
    %pip install -q POT
if INSTALL_PYMC:
    %pip install -q pymc

In [ ]:
# Part 1.2 — versions, the fp16 loss fix (static check, no monkey-patch), GPU / CUDA / memory / disk
import nflbowl
from nflbowl.research import colab

NFLBOWL_VERSION = colab.assert_version('0.3.0')     # 0.3.0 = first release with nflbowl.research (this notebook)
MISSING = colab.missing_required()
assert not MISSING, f'missing third-party packages {MISSING}: re-run Part 1.1'
FIX = colab.fixed_losses_check()
print(f'nflbowl {NFLBOWL_VERSION} from {nflbowl.__file__}')
print('fp32 contrastive / InfoNCE / MMD regions present (session-3 fix):', FIX['ok'], '·', {k: v for k, v in FIX.items() if k not in ('ok', 'source_file')})
ENV = colab.runtime_report()
if not ENV.get('cuda_available'):
    print('WARNING: no CUDA GPU. Warehouse stages run on CPU; 12F training would be very slow (Runtime → Change runtime type → GPU).')

In [ ]:
# Part 1.3 — Google Drive: inputs (bundle, warehouse) and every persistent output
DRIVE_OK = colab.mount_drive()
print('Drive mounted:', DRIVE_OK)

## Part 2 · Central experiment configuration

Everything the notebook does is decided here. `run_mode` only switches OPTIONAL analyses and the encoder preset; declared estimators, bootstrap sizes, folds and seeds of the primary analyses never change.

- **Precision.** `fp32` by default (the first Colab F4 run overflowed under fp16). `amp` = fp16 autocast + GradScaler (Colab T4) once Part 1.2 shows the fix; precision is part of every checkpoint signature, so fp32 and AMP checkpoints never mix.
- **Batch / gradient accumulation** are SCIENCE settings (`encoder_science_overrides`, e.g. `{'science.accum_steps': 2}`): they change the science hash, so they get their own experiment directory automatically.
- **Recovery.** Re-running the notebook with the same `experiment_name` reopens everything on Drive: finished stages are reused, interrupted training resumes from the last epoch checkpoint.
- **Seed.** `20261015` is the declared Milestone 12 seed (fold manifest, bootstrap draws, encoder initialisation). A different seed is a different experiment.

In [ ]:
CFG = {
    'experiment_name': 'm12_unified',
    'run_mode': 'fast',                  # 'debug' wiring check · 'fast' every primary analysis · 'full' every declared analysis
    'seed': 20261015,                    # declared Milestone 12 seed
    'resume': True,                      # reuse finished artifacts, resume interrupted training
    'run_full_research': False,          # True: all optional analyses + other encoder families, ablations, inductive encoders, learning curve
    # ---- locations
    'drive_root': '/content/drive/MyDrive/NFLBowl',
    'bundle_dir': '/content/drive/MyDrive/nfl-big-data-bowl-2027/bundles/m12f_v1',
    'bundle_zip': None,                  # or a path to m12f_v1.zip (extracted once)
    'research_root': '/content/drive/MyDrive/nfl-big-data-bowl-2027/research_root',   # Mode B: contains warehouse/ (and data/, reports/, artifacts/)
    'local_stage_dir': '/content/nflbowl_stage',
    'data_mode': 'auto',                 # 'auto' · 'bundle' (Mode A only) · 'warehouse' (Mode B only)
    'warehouse_run_mode': 'full',        # nflbdb RUN_MODE of the warehouse stages ('smoke' needs the _sample artifacts)
    'build_bundle_from_warehouse': False,
    'verify_bundle_sha256': False,
    # ---- GPU / 12F
    'device': 'auto',
    'precision': 'fp32',                 # 'fp32' · 'amp' (after the fix check) · 'bf16' (A100 / L4 only; not the notebook numerics)
    'embed_batch': 4096,
    'encoder_preset': None,              # None → colab_gpu (fast / full) or fast_iteration (debug)
    'encoder_science_overrides': {},     # e.g. {'science.accum_steps': 2} → own experiment directory
    'encoder_experiment_dir': None,      # e.g. '/content/drive/MyDrive/NFLBowl/experiments/f4_main' to continue an earlier run
    'run_gpu_training': True,
    'encoder_primary': 'F4',
    'encoder_baselines': ['F0', 'random'],
    'encoder_families_extra': [],        # e.g. ['F2', 'F3', 'F5', 'F6']
    'encoder_ablations': False,
    'encoder_inductive': False,          # F2 + F4 × 5 folds (hours): NFL transfer ladder + 12G encoder block
    'encoder_learning_curve': False,
    'encoder_eval_level': 'standard',
    # ---- persistence
    'save_figures': True, 'save_tables': True, 'save_models': True, 'save_processed_data': True, 'save_reports': True,
    'figure_formats': ['png', 'svg', 'pdf'], 'figure_dpi': 300,
    # ---- stages
    'stages': {'eda': True, '12.0': True, '12C': True, '12B': True, '12A': True, '12D': True, '12E': True, '12F': True, '12G': True},
    'reuse_completed_stages': True,
    'continue_on_error': True,           # a failing stage is recorded (logs/<stage>_traceback.txt); dependants gate themselves off
    'options': {},                       # e.g. {'12C': {'permutation': True}, '12E': {'heavy_methods': True}}
    'knobs': {},
}

In [ ]:
from pathlib import Path

from nflbowl.research import artifacts, data_access, eda, encoder, evaluation, reporting, stages
from nflbowl.research import config as unified_config
from nflbowl.research.stages import display_df, md, run_guarded
from nflbowl.utils.reproducibility import seed_everything

U = unified_config.resolve(CFG)
if U.precision in ('amp', 'fp16') and not FIX['ok']:
    raise RuntimeError('AMP requested but the installed code lacks the fp32 loss regions: keep precision = fp32 or reinstall (Part 1.1)')
RUN = artifacts.ResearchRun(U.run_dir, U, local_tmp=Path(U.local_stage_dir) / 'write_tmp')
U.save(RUN.path('config', 'unified_config.json'))
RUN.save_json({**ENV, 'fix_check': FIX}, 'environment', 'run', subdir='config')
seed_everything(U.seed)
RUN.log(f'run directory {U.run_dir} · run mode {U.run_mode} · precision {U.precision} · preset {U.encoder_preset}')
display_df(unified_config.describe(U), rows=200, strlen=110)

## Part 3 · Data access and validation

**Mode A — portable bundle** (`bundle.json`, `folds.parquet`, `sequence_store/{X,M}.npy`, `meta.parquet`, `COMPLETE.json`; optional `athletes.parquet`, `tcn_representations.parquet`): staged once to local disk (memory-mapped reads from Drive are slow), every schema validated, plus content checks (channels, athlete ids, fold range / coverage, both-domain athletes, empty sequences, athletes table, transfer target). It does NOT contain the Milestone 11 histograms, attempt features or envelopes — 12.0 and 12A–12E, 12G need Mode B.

**Mode B — research warehouse**: `research_root/warehouse/gold/advanced_movement/{discovery_v1, intelligence_v1, advanced_v1, transfer}`. One `WarehouseContext` (the exact objects every source notebook built) whose store also mirrors every output into `RUN_DIR`.

The stage plan below is the expected decision per stage (current files + outputs that enabled upstream stages will write). Each stage re-checks its inputs when it starts.

In [ ]:
# Part 3.1 — which input modes exist; expected decision per stage
SRC = data_access.discover(U)
display_df(SRC.table(), strlen=160)
PLAN = data_access.plan(U, SRC)
RUN.save_table(PLAN, 'stage_plan', 'data', subdir='diagnostics')
display_df(PLAN, strlen=160)

In [ ]:
# Part 3.2 — Mode A: stage, open and validate the bundle
from nflbowl.config import ScienceConfig

OUTER_FOLDS = ScienceConfig().outer_folds
BUNDLE, FOLD_TABLE = None, None
if SRC.bundle_dir is not None:
    from nflbowl.data.loading import open_bundle

    BUNDLE = open_bundle(SRC.bundle_dir, U.local_stage_dir, verify=U.verify_bundle_sha256, outer_folds=OUTER_FOLDS)
    BCHECKS, FOLD_TABLE = data_access.bundle_checks(BUNDLE, OUTER_FOLDS)
    RUN.save_table(BCHECKS, 'bundle_validation', 'data', subdir='diagnostics')
    RUN.save_processed_dataset(BUNDLE.summary(), 'bundle_summary', 'data', {'fingerprint': BUNDLE.fingerprint})
    display_df(BCHECKS, strlen=160)
    if (BCHECKS['status'] == 'fail').any():
        raise RuntimeError('bundle content checks failed (see diagnostics/bundle_validation.csv); rebuild the bundle (docs/nflbowl/BUNDLE_BUILD_GUIDE.md)')
    print(f'bundle {BUNDLE.fingerprint} · {BUNDLE.n_sequences:,} sequences · staged at {BUNDLE.root}')
else:
    md('_No bundle: 12F and the bundle part of the EDA are skipped (set `bundle_dir` / `bundle_zip`, or `build_bundle_from_warehouse` in Mode B)._')

In [ ]:
# Part 3.3 — Mode B: one warehouse context for every stage; input inventory; software checks (run once, synthetic data only)
WH, CHECKS = None, None
WAREHOUSE_STAGES = ('12.0', '12C', '12B', '12A', '12D', '12E', '12G')
if SRC.warehouse and any(U.enabled(s) for s in WAREHOUSE_STAGES):
    from nflbowl.research.warehouse import WarehouseContext

    WH = WarehouseContext.build(SRC.research_root, RUN, run_mode=U.warehouse_run_mode, seed=U.seed, save_tables=U.save_tables, save_figures=U.save_figures)
    display_df(WH.describe())
    INV = data_access.warehouse_inventory(SRC.research_root, U.warehouse_run_mode)
    RUN.save_table(INV, 'warehouse_inventory', 'data', subdir='diagnostics')
    display_df(INV, strlen=120)
    CHECKS = stages.run_software_checks(RUN)
    display_df(CHECKS, strlen=120)
else:
    md('_No research warehouse in this runtime: 12.0, 12A–12E and 12G are skipped with their reason (the portable bundle does not contain their inputs)._')

### Part 3.4 · 12.0 — research foundation and feasibility (warehouse)

**Question.** Which athletes, observations and speed states support each method, and on which shared grids, cohorts and folds will every method be compared? **Not a test** — an inventory every later stage checks (`fd.check_foundation`).

**Contract.** In: Milestone 11 officials, folds, attempt index, envelopes, NFL histograms (lazy), 09A episodes; Combine frames only when the curve tables are absent. Out: `m12_combine_curves / _speed_time / _hist / _quality`, `m12_nfl_bin_support`, `m12_game_top_speed`, `m12_shared_speed_support`, `m12_movement_grids`, `m12_cohort_registry`, `m12_experiment_availability`, `m12_fold_manifest`, `m12_foundation_manifest.json`. Cost: minutes (no tracking scan when the curve tables exist).

In [ ]:
S12_0 = stages.Foundation(WH, RUN, U, SRC, CHECKS)
S12_0.explain()
S12_0.inventory()
S12_0.curve_tables()

In [ ]:
S12_0.definitions()      # shared speed range (support counts only), grids, cohort registry, folds
S12_0.baselines()

In [ ]:
S12_0.figures()
S12_0.conclude()
S12_0.finish()

## Part 4 · Unified exploratory analysis

One EDA instead of eight partial ones: athlete coverage (Combine / NFL / both) by position, sequences per domain × movement kind, valid sequence length and peak speed, EXACT movement-signal distributions by domain (accumulated chunk by chunk over every valid frame; no sampling), sampling-interval and heading-validity quality, fold balance per position, missingness of official measurements, relationships between physical measurements, the NFL target by position, and a leakage-risk checklist; with the warehouse, registry coverage and NFL exposure. The 12.0 funnel, repeat availability, position support, speed support and feasibility figures were drawn in Part 3.4. Method-specific diagnostics stay in their stage. Reloads its exact statistics from `RUN_DIR` on a rerun.

In [ ]:
def run_eda():
    tables, notes = [], []
    if BUNDLE is not None:
        freeze = data_access.read_json(data_access.gold_dir(SRC.research_root, U.warehouse_run_mode) / 'advanced_v1' / 'm12c_target_freeze.json') if SRC.warehouse else None
        tables += eda.run_bundle_eda(RUN, BUNDLE, OUTER_FOLDS, freeze, FOLD_TABLE, reuse=U.resume)
        notes.append(f'bundle {BUNDLE.fingerprint}: {BUNDLE.n_sequences:,} sequences, {BUNDLE.meta["nfl_id"].n_unique():,} athletes')
    else:
        notes.append('no bundle: bundle part not run')
    if WH is not None and WH.store.has('m12_cohort_registry'):
        tables += eda.run_foundation_eda(RUN, WH)
        notes.append('warehouse: 12.0 cohort registry')
    else:
        notes.append('no 12.0 cohort registry: foundation part not run')
    RUN.save_markdown(reporting.eda_report(RUN, tables, notes), 'eda_unified', 'eda')
    return tables


EDA_TABLES = run_guarded(RUN, U, 'eda', run_eda, enabled=U.enabled('eda') and (BUNDLE is not None or WH is not None), reason='disabled or no input data')

## Part 5 · 12C — distributional NFL movement capacity (highest priority; defines the target)

**Question (Q3).** Can stable athlete-specific NFL speed-conditioned acceleration distributions be estimated more reliably than noisy maxima or fitted linear envelopes — and do Combine speed features predict them beyond official testing?

**H1 (measurement, primary).** Upper conditional acceleration quantiles (τ = 0.90) over the 12.0 shared speed range are more game-split reliable than the observed maximum (C0a) and the linear envelope (C0b). **H2 (transfer, primary).** M3 (M1 + fitted sprint Vmax) lowers held-out RMSE of `nfl.q90_shared` vs M1; M2 (M1 + observed top speed) lowers held-out RMSE of `nfl.game_top_speed_q75` vs M1; Holm over the two.

**Target freeze.** The estimator behind `nfl.q90_shared` is chosen by a declared reliability rule on NFL data ONLY and written before any Combine value is joined; it is immutable. **Not claimed:** an observed upper quantile is what the athlete did under game opportunity, not a physiological maximum.

**Contract.** In: 12.0 outputs, `m11_envelope_hist`, `m11_envelopes`, `m11_athlete_features`, `m11_nfl_targets`, officials (+ 09A episodes for C7). Out: `m12c_*` tables, `m12c_target_freeze.json`, `m12c_targets`, figures `12c_*`, report `12c_distributional_capacity.md`. Cost: minutes; the permutation null and the C7 replication grid are options (`full` mode).

In [ ]:
S12C = stages.DistributionalCapacity(WH, RUN, U, SRC, CHECKS)
S12C.explain()
S12C.prepare()           # eligibility, memory plan, athlete cubes (games = replication unit)
S12C.estimators()        # C2 penalty on the dedicated tuning split; every estimator × τ
S12C.athlete_example()   # one rule-chosen athlete; position quantile curves

In [ ]:
S12C.reliability()       # PRIMARY H1: game-split reliability, differences vs C0a / C0b
S12C.forecasts()         # held-out-game pinball, coverage, CRPS
S12C.freeze()            # declared rule on NFL data only → immutable freeze → athlete targets

In [ ]:
S12C.transfer()                 # PRIMARY H2: ladder M0–M6 on identical athletes per target, Holm
S12C.controls_and_subgroups()   # within-position permutation (option), by position, M1+ sensitivity
S12C.replication()              # C7: the two 11B high-speed findings under declared variants (option)
S12C.uncertainty()              # exposure sensitivity, per-athlete trait SE

In [ ]:
S12C.figures()
S12C.conclude()
S12C.finish()

## Part 6 · 12B — measurement error and latent traits

**Question (Q2).** Are Combine → NFL relationships attenuated because tracked Combine predictors are measured unreliably — are earlier null results compatible with realistic attenuation?

**Four estimands, never substituted for each other:** observed association (partial r / standardized slope of the athlete mean given M1) · corrected LATENT effect (regression calibration, SIMEX, joint EM; disattenuated r) · compatible latent range · out-of-sample value (held-out Δ RMSE of M1 + x̄ and M1 + E[θ | x̄]). Observed performance is never read as latent ability. **Assumptions (stated, not tested):** classical, nondifferential, homoscedastic error; exchangeable attempts; with one attempt σ²_ε is not identifiable (flagged, never borrowed).

**Contract.** In: 12.0, `m11_attempt_features`, `m11_nfl_targets`, `m11_nfl_target_reliability` (+ 12C target when present). Out: `m12b_*`, figures `12b_*`, report `12b_measurement_error.md`. Cost: seconds to minutes; the checkpointed synthetic grid (resumable) and PyMC are options.

In [ ]:
S12B = stages.MeasurementError(WH, RUN, U, SRC, CHECKS)
S12B.explain()
S12B.prepare()
S12B.reliability()       # variance components, ICC by position and attempts, Spearman–Brown

In [ ]:
S12B.corrections()       # B0–B3 estimands + held-out B0 / B1 on identical athletes (RC fitted on training athletes)
S12B.synthetic()         # planted-signal recovery grid (option; checkpointed per scenario)
S12B.bayes()             # optional PyMC model
S12B.figures()
S12B.conclude()
S12B.finish()

## Part 7 · 12A — functional movement transfer

**Question (Q1).** Does modelling the whole Combine acceleration–speed function a_C(v) retain transferable information that scalar band features lose — and where along the speed axis does transfer sit?

**Primary.** A2 (M1 + fold-fitted FPCA scores of a_C(v)) lowers the held-out athlete-mean squared error of the NFL τ-quantile function a_N(v) (the 12C frozen estimator) vs A0 (M1 + scalar bands), identical athletes. **Secondary:** A1 P-splines, A4 hierarchical shrinkage, pointwise profile, controls (shuffled curves, random scores), FPC test–retest; elastic phase / amplitude alignment (exploratory option).

**Fixed rules.** Shared speed range only; each athlete scored on its own supported bins; ≤ 1 interior gap bridged; no extrapolation; every cross-athlete quantity fitted on training athletes; the spline penalty chosen by Combine repeat agreement (no NFL value); registration never discards timing.

**Contract.** In: 12.0 + the **12C freeze (hard gate)**, `m12_combine_curves` (+ speed–time curves, twins for the elastic part). Out: `m12a_*`, figures `12a_*`, report `12a_functional_transfer.md`. Cost: minutes; elastic alignment tens of minutes.

In [ ]:
S12A = stages.FunctionalTransfer(WH, RUN, U, SRC, CHECKS)
S12A.explain()
S12A.prepare()
S12A.representations()   # PRIMARY: A2 vs A0 (function-on-function) + pointwise mode + controls

In [ ]:
S12A.localisation()      # pointwise effect along speed, positions, hierarchical repeat prediction, FPC test–retest
S12A.elastic()           # exploratory option: landmark + SRSF alignment, phase / amplitude, twins
S12A.figures()
S12A.conclude()
S12A.finish()

## Part 8 · 12D — multi-view latent athletic traits

**Question (Q4).** Do several Combine drills, treated as measurement views, jointly reveal reproducible physical traits that single drill metrics measure unreliably?

**Primary (reliability).** Cross-fitted shared factor scores have higher attempt-split test–retest reliability than the most reliable single feature loading on them. **Secondary (transfer).** Factor scores add held-out information beyond official tests (D0) and the concatenated features (D1).

**Views / latent assumptions.** Views = tracked drill families (≤ 4 physics-preselected features each); official tests are D0, never a view. D2 probabilistic FA · D3 structured multi-view FA (shared + view-specific) · D4 D3 on position-centred features · D5 only if its support rule passes. Views are missing NOT at random: scores use observed views only, missingness is diagnosed, complete cases are a sensitivity, factors are not named without loading and stability support.

**Contract.** In: 12.0, `m11_athlete_features`, `m11_attempt_features`, `m11_nfl_targets`, `m11_envelopes` (+ 12C target). Out: `m12d_*`, `m12d_selected_representation.json` (declared rule, no NFL value; used by 12C-M6 and 12G), figures `12d_*`, report `12d_multiview_traits.md`. Cost: minutes; bootstrap stability is an option.

In [ ]:
S12D = stages.MultiviewTraits(WH, RUN, U, SRC, CHECKS)
S12D.explain()
S12D.prepare()           # views, availability, missingness diagnostics
S12D.dimensionality()    # K by held-out log-likelihood (no NFL value)
S12D.reliability()       # PRIMARY: cross-fitted test–retest vs best single feature; stability; selection rule

In [ ]:
S12D.transfer()          # secondary: D0 / D1 / D2–D4 on identical athletes per target
S12D.figures()
S12D.conclude()
S12D.finish()

## Part 9 · 12E — conditional optimal transport matching

**Question (Q5).** Do comparisons of whole movement DISTRIBUTIONS recover cross-domain athlete similarity that point features miss?

**Primary.** E5 (per shared speed bin, W1 between accelerating-state acceleration distributions; mean over bins supported in both domains) gives a higher linked-athlete retrieval AUC than the standardized feature distance E0, paired over athletes, AND E5's AUC interval lies above 0.5. **Transport constraints:** one physical grid for both domains (shared speed bins × 0.5 yd/s² over −4…6 yd/s²), comparable state = accelerating, same-position candidate pools identical for every method, E2 exact only with POT (never substituted), E4 / E6 entropic (ε relative to the median cost), E6 on unnormalised frame shares.

**Restored definitions.** The committed 12E notebook had replaced the module definitions with inline ones and dropped the secondary analyses; this part uses the declared `movement_transport` definitions (see the merge matrix). Outputs carry `definitions = module_v1`.

**Contract.** In: 12.0, `m12_combine_hist`, `m12_combine_quality`, `m11_envelope_hist`, `m11_athlete_features`, `m11_nfl_targets` (+ 12C targets for the downstream ladder, twins). Out: `m12e_*`, figures `12e_*`, report `12e_optimal_transport.md`. Cost: minutes for E0/E1/E3/E5; E4 / E6 / ε grid / per-athlete bootstrap are options (tens of minutes).

In [ ]:
S12E = stages.OptimalTransport(WH, RUN, U, SRC, CHECKS)
S12E.explain()
S12E.prepare()
S12E.retrieval()          # PRIMARY: E0–E6 retrieval AUC within position; paired Δ AUC vs E0

In [ ]:
S12E.negative_controls()  # shuffled identities, coarsened bins, uniform mass on support, ε sensitivity (option)
S12E.consistency()        # within-domain identification: NFL odd / even games, Combine attempt 1 / 2
S12E.uncertainty()        # per-athlete bootstrap of the own distance (option)
S12E.twins()
S12E.downstream()         # M1 vs M1 + distance to the training-fold position reference

In [ ]:
S12E.figures()
S12E.conclude()
S12E.finish()

## Part 10 · 12F — physics-informed shared movement encoder (GPU)

**Question (Q6).** Can an encoder trained on BOTH Combine drills and NFL movement windows learn a representation that preserves interpretable physics and transfers across domains better than engineered features and the Combine-only TCN-B?

**Primary (declared).** The INDUCTIVE F4 encoder beats the random encoder AND F0 on held-out cross-domain retrieval, and M1 + its fold embeddings lowers held-out RMSE of the 12C target vs M1. With the defaults only TRANSDUCTIVE F4 is trained (label-free diagnostics); the inductive test needs `encoder_inductive = True` (hours).

**Engineering.** `nflbowl.Experiment` on the bundle: same `se.batch_loss`, splits and estimands as the 12F notebook; GPU-resident data; precision from CFG (fp32 default; AMP optional); contrastive / InfoNCE / MMD in fp32 regions; per-epoch checkpoints (model, optimizer, scheduler, GradScaler, RNG) on Drive; a finished run is restored, an interrupted one resumes. Training (GPU) and evaluation (CPU-heavy) are separate phases. F4 is the only encoder trained by default.

**Contract.** In: bundle (+ `athletes.parquet` with `nfl.q90_shared` for the ladder, `tcn_representations.parquet` for F1). Out: `checkpoints/experiments/<name>/…`, `embeddings/`, tables `12f_*`, figures `12f_*`, report `12f_shared_encoder.md`; with a warehouse also `m12f_*` + the 12F run manifest so 12G includes 12F.

In [ ]:
ENC = encoder.EncoderSection(RUN, U, SRC, WH)
ENC.explain()
ENC.open()               # new or reopened experiment under RUN_DIR/checkpoints/experiments
ENC.prepare()            # stage + validate bundle, derived targets, F0 features (cached)

In [ ]:
# GPU training — F4 (primary). After a Colab disconnect: rerun Parts 1–3 and this cell; it resumes from the last epoch checkpoint.
ENC.train_primary()

In [ ]:
# opt-in (CFG): other families / ablations, inductive F2 + F4 per fold, learning curve; then any job a disconnect left unfinished
ENC.train_extras()
ENC.train_inductive()
ENC.train_learning_curve()
ENC.resume_unfinished()

In [ ]:
# evaluation after all GPU work: embeddings (reused when saved), label-free metrics, inductive retrieval, transfer ladder
ENC.embed_and_evaluate()
ENC.transfer()

In [ ]:
ENC.collect()                # figures, evaluation views, embeddings, checkpoint metadata → RUN_DIR
ENC.publish_to_warehouse()   # 12F comparisons / evidence / run manifest for 12G (warehouse only)
ENC.conclude()
ENC.finish()

## Part 11 · Unified scientific evaluation

The comparison rows of every executed stage are read back from `RUN_DIR` and classified — never pooled: **diagnostic** (software checks, synthetic recovery, controls) · **measurement** (ICC, split-half, test–retest, within-domain identification, held-out forecast quality) · **transductive** (label-free: 12F transductive encoders, 12E retrieval) · **in-sample estimand** (12B coefficients) · **inductive** (cross-fitted held-out ladders); `outcome_linked` flags rows involving an NFL target. Families with a comparable estimand are put side by side with an explicit `estimand` column: cross-domain retrieval (12E distributions vs 12F embeddings), identity controls, domain shift, physical probes, latent-trait consistency, calibration, ablations, folds. Intervals stay the stages' paired / athlete bootstrap intervals. Only the 12G tournament compares methods on identical athletes.

In [ ]:
def run_evaluation():
    tabs = evaluation.build(RUN)
    evaluation.figures(RUN, tabs)
    for k in ('model_comparison', 'cross_domain_retrieval', 'athlete_identity_controls', 'domain_shift_diagnostics', 'latent_trait_consistency'):
        md(f'**{k}**')
        display_df(tabs.get(k), strlen=110)
    RUN.save_markdown(reporting.evaluation_report(RUN, tabs), 'unified_evaluation', 'evaluation')
    return tabs


EVAL = run_guarded(RUN, U, 'evaluation', run_evaluation) or {}

## Part 12 · 12G — research synthesis and decisions

**Question (Q7).** Which advanced methods outperform simpler physical models under identical cohorts, baselines and protocol — and what next? Evidence registry, the six declared experiments, the evidence-status matrix (text statuses, never a combined score), the common-cohort tournament (exploratory; Holm / BH; permutation of the best block; cohort-size sensitivity as an option), rule-based recommendations. The 12F encoder block uses the warehouse tables written in Part 10, or the inductive F4 fold embeddings directly. Missing tracks are `execution_pending`, never negative results.

**Decisions.** The decision matrix maps each track's DECLARED candidate use for scouts, general managers, coaches and player development to this run's primary evidence: supported → candidate use pending independent confirmation; inconclusive → descriptive only; reversed → do not use. Nothing here is causal.

In [ ]:
S12G = stages.Synthesis(WH, RUN, U, SRC, CHECKS)
S12G.explain()
S12G.registry()
S12G.experiments()
S12G.tournament(encoder_byf=ENC.inductive_byf())

In [ ]:
S12G.interpretation()
S12G.figures()
S12G.conclude()
DEC = reporting.decision_matrix(RUN, EVAL.get('unified_comparisons'))
RUN.save_table(DEC, 'decision_matrix', '12G')
display_df(DEC, strlen=200)
S12G.add_report_table('Decisions for scouts, general managers, coaches and player development (declared uses × primary evidence of this run)', DEC)
S12G.finish()

## Part 13 · Artifact inventory and final report

`reports/markdown/FINAL_RESEARCH_REPORT.md` (+ HTML): executed / reused / skipped / failed stages, data sources, model configurations, validation protocols, metrics, what the data support, decisions, saved artifacts, limitations, next directions — generated from this run's persisted tables. Then the stage records, the artifact inventory and a ZIP export (config, manifests, tables, figures, reports, diagnostics, logs) on Drive.

In [ ]:
STAGE_OBJECTS = [S12_0, S12C, S12B, S12A, S12D, S12E, ENC, S12G]
FINAL = reporting.final_report(RUN, U, SRC, EVAL, DEC, STAGE_OBJECTS)
RUN.save_markdown(FINAL, 'FINAL_RESEARCH_REPORT', 'final')
md(FINAL)

In [ ]:
display_df(RUN.stage_table(), strlen=200)
INVENTORY = RUN.inventory()
RUN.save_table(INVENTORY, 'artifact_inventory', 'final', subdir='manifests')
display_df(INVENTORY.group_by('area').len().sort('area'))
ZIP = RUN.export_zip()
print(f'run directory: {RUN.root}')
print(f'export: {ZIP}')